In [ ]:
import os
import time
import requests
import sqlite3
from dotenv import load_dotenv

# cargar API key
load_dotenv()
api_key = os.getenv("API_KEY")

# Configuración del muestreo
TARGET_MATCHES_PER_TIER = 15000         # Objetivo de partidas unicas por cada Elo
TIERS_TO_EXTRACT = ["EMERALD", "DIAMOND"]
RANKS = ["IV", "III", "II", "I"]

# Fechas del parche y cola de SoloQ (420)
START_TIME = 1789000200 # 10/09/2026
END_TIME = 1790123400   # 22/09/2026
QUEUE_SOLOQ = 420

# Base de Datos SQLite
conn = sqlite3.connect("data/riot_dataset.db")
cursor = conn.cursor()

# Tabla de partidas (guardamos el tier para poder contar cuántas llevamos de cada uno)
cursor.execute('''
    CREATE TABLE IF NOT EXISTS matches (
        match_id TEXT PRIMARY KEY,
        tier TEXT,
        processed INTEGER DEFAULT 0
    )
''')
# Tabla de seguimiento para no repetir llamadas a jugadores ya consultados
cursor.execute('''
    CREATE TABLE IF NOT EXISTS scanned_puuids (
        puuid TEXT PRIMARY KEY
    )
''')
conn.commit()

def request_riot(url):
    while True:
        resp = requests.get(url)

        match resp.status_code:
            case 429:
                wait_time = int(resp.headers.get('Retry-After', 5))
                print(f'Límite de API excedido. Esperando {wait_time}s...')
                time.sleep(wait_time + 1)
                continue
            case 404:
                return None
            case 401:
                print("La API Key ha expirado.")
                api_key_expired = True
                break
            case 200:
                return resp.json()
            case _:
                print(f"Error {resp.status_code} al consultar {url}")
                return None

base_url_euw = "https://euw1.api.riotgames.com/lol"
base_url_matches = "https://europe.api.riotgames.com/lol/match/v5/matches/by-puuid/"

for tier in TIERS_TO_EXTRACT:
    # Verificamos si ya habia partidas descargadas de este Elo
    cursor.execute("SELECT count(*) FROM matches WHERE tier = ?", (tier,))
    matches_count = cursor.fetchone()[0]
    
    print(f"\n=======================================================")
    print(f" Iniciando {tier} | Progreso actual: {matches_count}/{TARGET_MATCHES_PER_TIER}")
    print(f"=======================================================")
    
    # Cada rango (IV, III, II, I) mantiene su propio numero de pagina independiente
    pages = {rank: 1 for rank in RANKS}
    active_ranks = list(RANKS)
    
    # Mientras no lleguemos al objetivo y queden paginas con jugadores
    while matches_count < TARGET_MATCHES_PER_TIER and active_ranks:
        
        # Round-robin: pide una página de IV, luego III, II, I...
        for rank in list(active_ranks):
            if matches_count >= TARGET_MATCHES_PER_TIER:
                break
                
            current_page = pages[rank]
            league_url = f"{base_url_euw}/league-exp/v4/entries/RANKED_SOLO_5x5/{tier}/{rank}?page={current_page}&api_key={api_key}"
            players = request_riot(league_url)
            pages[rank] += 1
            
            # Si no devuelve jugadores, este rango ya no tiene más páginas
            if not players:
                print(f"Fin de jugadores disponibles en {tier} {rank}.")
                active_ranks.remove(rank)
                continue
                
            # Procesamos los jugadores obtenidos en esta pagina
            for player in players:
                if matches_count >= TARGET_MATCHES_PER_TIER:
                    break
                    
                puuid = player.get('puuid')
                if not puuid:
                    continue
                
                # Comprobamos si ya leimos a este jugador antes
                cursor.execute("SELECT 1 FROM scanned_puuids WHERE puuid = ?", (puuid,))
                if cursor.fetchone():
                    continue
                
                # Buscamos las partidas del jugador en ese parche
                start_idx = 0
                while True:
                    matches_url = f"{base_url_matches}{puuid}/ids?startTime={START_TIME}&endTime={END_TIME}&queue={QUEUE_SOLOQ}&start={start_idx}&count=100&api_key={api_key}"
                    match_ids = request_riot(matches_url)
                    
                    if not match_ids:
                        break
                        
                    for match_id in match_ids:
                        # INSERT OR IGNORE descarta duplicados
                        cursor.execute("INSERT OR IGNORE INTO matches (match_id, tier) VALUES (?, ?)", (match_id, tier))
                        
                        # cursor.rowcount es 1 solo si la partida era nueva y se inserto
                        if cursor.rowcount > 0:
                            matches_count += 1
                            if matches_count >= TARGET_MATCHES_PER_TIER:
                                break
                                
                    conn.commit()
                    
                    # Si devolvio menos de 100 o ya alcanzamos la meta, salimos de este jugador
                    if len(match_ids) < 100 or matches_count >= TARGET_MATCHES_PER_TIER:
                        break
                    start_idx += 100
                
                # Marcamos al jugador como escaneado
                cursor.execute("INSERT OR IGNORE INTO scanned_puuids (puuid) VALUES (?)", (puuid,))
                conn.commit()

            print(f"[{tier}] Progreso: {matches_count}/{TARGET_MATCHES_PER_TIER} partidas únicas | Ronda completada para {tier} {rank} (Pág {current_page})")

print("\n¡Proceso de recolección de IDs finalizado con éxito!")

# Resumen global de la base de datos
cursor.execute("SELECT tier, count(*) FROM matches GROUP BY tier")
resumen = cursor.fetchall()
for t, total in resumen:
    print(f"- {t}: {total} partidas únicas registradas")


 Iniciando EMERALD | Progreso actual: 0/15000


'Límite de API excedido. Esperando 77s...'

KeyboardInterrupt: 

In [3]:
import os
import time
import json
import requests
import sqlite3
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("API_KEY")

conn = sqlite3.connect("data/riot_dataset.db")
cursor = conn.cursor()

base_url_match = "https://europe.api.riotgames.com/lol/match/v5/matches/"

# Cargamos a la RAM solo las partidas que nos faltan por descargar
cursor.execute("SELECT match_id FROM matches WHERE processed = 0")
matches_pendientes = [row[0] for row in cursor.fetchall()]

print(f"Iniciando descarga de {len(matches_pendientes)} partidas pendientes...")

api_key_expired = False
partidas_guardadas = 0

# Abrimos el JSONL en modo append ("a") para ir añadiendo sin sobreescribir
with open("data/matches_dataset.jsonl", "a", encoding="utf-8") as f_out:
    for idx, match_id in enumerate(matches_pendientes):
        if api_key_expired:
            break

        url = f"{base_url_match}{match_id}?api_key={api_key}"
        
        while True:
            resp = requests.get(url)
            
            match resp.status_code:
                case 429:
                    wait_time = int(resp.headers.get('Retry-After', 5))
                    print(f'Límite de API excedido. Esperando {wait_time}s...')
                    time.sleep(wait_time + 1)
                    continue
                case 401:
                    print("La API Key ha expirado.")
                    api_key_expired = True
                    break
                case 200:
                    data = resp.json()
                    # Python permite usar .startswith("16.18") independientemente de lo que siga
                    game_version = data.get("info", {}).get("gameVersion", "")
                    
                    if game_version.startswith("16.18"):
                        f_out.write(json.dumps(data) + "\n")
                        partidas_guardadas += 1
                        # 1 = Descargada y guardada correctamente
                        cursor.execute("UPDATE matches SET processed = 1 WHERE match_id = ?", (match_id,))
                    else:
                        # 2 = Descargada pero descartada (es de otro parche, ej: 16.17)
                        cursor.execute("UPDATE matches SET processed = 2 WHERE match_id = ?", (match_id,))
                    break
                case _:
                    print(f"Error {resp.status_code} en partida {match_id}. Descartando.")
                    # -1 = Error (para no intentarlo en el siguiente bucle infinito)
                    cursor.execute("UPDATE matches SET processed = -1 WHERE match_id = ?", (match_id,))
                    break
        
        # Hacemos commit en la base de datos en lotes para no saturar el disco
        if (idx + 1) % 50 == 0:
            conn.commit()
            print(f"Progreso: {idx + 1}/{len(matches_pendientes)} procesadas | {partidas_guardadas} partidas válidas guardadas.")

# Commit final para las últimas partidas que quedaron en el tintero
conn.commit()
print(f"\nProceso finalizado. Se han añadido {partidas_guardadas} nuevas partidas al dataset.")

Iniciando descarga de 687 partidas pendientes...
Límite de API excedido. Esperando 24s...
Progreso: 50/687 procesadas | 49 partidas válidas guardadas.


KeyboardInterrupt: 